# White-box jailbreak probe — results

Run the pipeline from the repo root (`jailbreak-probe/`) so paths resolve:

1. `python data/prepare_dataset.py`
2. `python features/extract_hidden_states.py`
3. `python probes/train_probes.py`
4. `python eval/evaluate.py`

Then reload this notebook.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path("..").resolve()
summary_path = ROOT / "probes" / "artifacts" / "train_summary.json"
metrics_path = ROOT / "results" / "metrics.json"

with open(summary_path, encoding="utf-8") as f:
    summary = json.load(f)

aucs = summary["val_auroc_per_layer"]
layers = np.arange(len(aucs))

plt.figure(figsize=(9, 4))
plt.plot(layers, aucs, marker="o", linewidth=1.5)
plt.xlabel("Layer index (0 = embeddings)")
plt.ylabel("Validation AUROC")
plt.title("Layer-wise linear probe (single layer per depth)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Best single layer:", summary["best_single_layer_idx"], summary["best_single_layer_auroc"])
print("Final layer val AUROC:", summary["val_auroc_final_layer"])
print("Multi-concat val AUROC:", summary["val_auroc_multi_concat"])

In [ ]:
if metrics_path.exists():
    with open(metrics_path, encoding="utf-8") as f:
        metrics = json.load(f)
    print("Test per-layer AUROC (first 5):", metrics["per_layer_test_auroc"][:5], "...")
    print("Test probe types:", metrics.get("probe_types", {}))
    if "early_token_multi_auroc" in metrics:
        print("Early-token (multi-probe) AUROC:", metrics["early_token_multi_auroc"])
    if "paraphrase_robustness" in metrics:
        print("Paraphrase robustness:", metrics["paraphrase_robustness"])
else:
    print("Run eval/evaluate.py to create", metrics_path)